## Análisis de datos - Clase 3
---

## Repaso práctico parte 1 - análisis inicial de un dataset
---


Vamos a realizar una actividad con un dataset con datos del Servicio Meteorológico Nacional Argentino.

El dataset contiene estadísticas climáticas normales (período 1991-2020) y temperaturas máximas y mínimas registradas durante el último año (365 días).

Fuente: https://www.smn.gob.ar/descarga-de-datos


Resumen de variables:
* estación, provincia, latitud, longitud, altitud, número y NroOACI (estos dos últimos identifican las estaciones)
* mes
* temperatura media, máxima y mínima
* humedad relativa
* velocidad del viento
* nubosidad total (en meteorología, la nubosidad se mide en octas (0, despejado - 8 completamente cubierto), que representan la fracción del cielo cubierta por nubes)
* precipitación
* frecuencia de días con precipitación superior a 0,1 mm.

In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy

In [2]:
smn_historico = pd.read_csv('../datasets/smn/smn_historico.csv', na_values=['S/D'])

In [3]:
# NEA: MISIONES, CORRIENTES, FORMOSA, CHACO

smn_historico.describe()

,altura,numero,LAT_decimal,LON_decimal,prec_sup_1mm,humedad,nubosidad,prec_mm,temp,temp_max,temp_min,viento
count,1152.000000,1152.000000,1152.000000,1152.000000,1061.000000,1110.000000,1166.000000,1061.000000,1167.000000,1158.000000,1124.000000,708.000000
mean,328.812500,87570.031250,-36.347049,-62.623785,5.097455,68.005405,3.840137,68.234873,14.834190,21.223834,9.102847,13.011582
std,468.195292,450.772923,10.369416,5.591299,2.536586,11.027475,0.888284,51.674519,8.178787,8.938676,7.778743,5.554040
min,5.000000,87007.000000,-77.866667,-72.050000,0.000000,25.700000,1.300000,0.000000,-20.900000,-17.500000,-25.000000,3.900000
25%,45.250000,87284.250000,-38.437500,-66.458333,3.100000,61.325000,3.300000,23.300000,10.300000,17.025000,4.500000,9.200000
50%,141.000000,87520.500000,-34.500000,-63.183333,5.200000,70.100000,3.700000,58.900000,15.700000,22.400000,9.900000,11.800000
75%,450.000000,87697.750000,-29.891667,-58.775000,7.000000,76.400000,4.200000,106.500000,21.200000,27.900000,15.100000,15.900000
max,3459.000000,89066.000000,-22.100000,-34.616667,13.800000,90.200000,7.500000,277.900000,28.300000,35.500000,22.500000,36.500000


In [4]:
smn_historico.columns

Index(['estacion', 'provincia', 'altura', 'numero', 'NroOACI', 'LAT_decimal',
       'LON_decimal', 'mes_txt', 'prec_sup_1mm', 'humedad', 'nubosidad',
       'prec_mm', 'temp', 'temp_max', 'temp_min', 'viento'],
      dtype='str')

In [24]:
valores_a_filtrar = ['MISIONES', 'CORRIENTES', 'FORMOSA', 'CHACO']
df = smn_historico[smn_historico['provincia'].isin(valores_a_filtrar)]
df.groupby(['provincia','estacion']).count()

altura  numero  NroOACI  \
provincia  estacion                                                     
CHACO      PRESIDENCIA ROQUE SAENZ PEÑA AERO      12      12       12   
           RESISTENCIA AERO                       12      12       12   
CORRIENTES CORRIENTES AERO                        12      12       12   
           MONTE CASEROS AERO                     12      12       12   
           PASO DE LOS LIBRES AERO                12      12       12   
FORMOSA    FORMOSA AERO                           12      12       12   
           LAS LOMITAS                            12      12       12   
MISIONES   BERNARDO DE IRIGOYEN AERO              12      12       12   
           IGUAZU AERO                            12      12       12   
           OBERA                                  12      12       12   
           POSADAS AERO                           12      12       12   

                                              LAT_decimal  LON_decimal  \
provincia  estacion                                                      
CHACO      PRESIDENCIA ROQUE SAENZ PEÑA AERO           12           12   
           RESISTENCIA AERO                            12           12   
CORRIENTES CORRIENTES AERO                             12           12   
           MONTE CASEROS AERO                          12           12   
           PASO DE LOS LIBRES AERO                     12           12   
FORMOSA    FORMOSA AERO                                12           12   
           LAS LOMITAS                                 12           12   
MISIONES   BERNARDO DE IRIGOYEN AERO                   12           12   
           IGUAZU AERO                                 12           12   
           OBERA                                       12           12   
           POSADAS AERO                                12           12   

                                              mes_txt  prec_sup_1mm  humedad  \
provincia  estacion                                                            
CHACO      PRESIDENCIA ROQUE SAENZ PEÑA AERO       12            12       12   
           RESISTENCIA AERO                        12            12       12   
CORRIENTES CORRIENTES AERO                         12            12       12   
           MONTE CASEROS AERO                      12            12       12   
           PASO DE LOS LIBRES AERO                 12            12       12   
FORMOSA    FORMOSA AERO                            12            12       12   
           LAS LOMITAS                             12            12       12   
MISIONES   BERNARDO DE IRIGOYEN AERO               12            12       12   
           IGUAZU AERO                             12            12       12   
           OBERA                                   12             5        8   
           POSADAS AERO                            12            12       12   

                                              nubosidad  prec_mm  temp  \
provincia  estacion                                                      
CHACO      PRESIDENCIA ROQUE SAENZ PEÑA AERO         12       12    12   
           RESISTENCIA AERO                          12       12    12   
CORRIENTES CORRIENTES AERO                           12       12    12   
           MONTE CASEROS AERO                        12       12    12   
           PASO DE LOS LIBRES AERO                   12       12    12   
FORMOSA    FORMOSA AERO                              12       12    12   
           LAS LOMITAS                               12       12    12   
MISIONES   BERNARDO DE IRIGOYEN AERO                 12       12    12   
           IGUAZU AERO                               12       12    12   
           OBERA                                      8        5     8   
           POSADAS AERO                              12       12    12   

                                              temp_max  temp_min  viento  
provincia  estacion                                   

In [25]:
df['altura'].mean()

np.float64(184.9090909090909)

In [26]:
df['temp'].mean()

np.float64(21.13671875)

In [27]:
df['temp'].std()

np.float64(4.0613963522075025)

In [37]:
# Corrección: Agrupamos por mes para calcular la precipitación MEDIA, como pide el enunciado
prec_media_por_mes = df.groupby('mes_txt')['prec_mm'].mean()

# Encontramos el mes con la mayor media y su valor
mes_maximo = prec_media_por_mes.idxmax()
valor_maximo = prec_media_por_mes.max()

print(f"El mes con mayor precipitación media es {mes_maximo} con {valor_maximo:.2f} mm")

'Oct'

In [42]:

df['mes_txt'].values

<bound method NDArrayBackedExtensionArray.unique of <StringArray>
['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct',
 ...
 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
Length: 132, dtype: str>

In [60]:
temperatura_medias_df = df.groupby('provincia')['temp'].mean().reset_index()

temperatura_medias_df.rename(columns={'temp': 'temp media'}, inplace=True)
temperatura_medias_df

,provincia,temp media
0,CHACO,21.295833
1,CORRIENTES,20.352778
2,FORMOSA,22.720833
3,MISIONES,20.827273


In [ ]:
temperatura_medias_est_df = df.groupby('estaciones')['temp'].mean().reset_index()

temperatura_medias_est_df.rename(columns={'temp': 'temp media'}, inplace=True)
temperatura_medias_est_df

# Reemplaza 'altura' y 'temp media' por los nombres exactos de tus columnas
coeficiente_spearman = temperatura_medias_est_df['altura'].corr(df['temp media'], method='spearman')

print(f"El coeficiente de Spearman es: {coeficiente_spearman:.4f}")

In [64]:
import pandas as pd

def print_outliers(df, var):
    # 1. Calcular el primer (Q1) y tercer cuartil (Q3)
    Q1 = df[var].quantile(0.25)
    Q3 = df[var].quantile(0.75)
    
    # 2. Calcular el Rango Intercuartílico (IQR)
    IQR = Q3 - Q1
    
    # 3. Definir los límites inferior y superior
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    # 4. Encontrar los outliers (mostrar solo los datos atípicos)
    outliers = df[(df[var] < limite_inferior) | (df[var] > limite_superior)]
    print("Outliers encontrados:")
    print(outliers)

print_outliers(df, 'temp')
print_outliers(df, 'prec_mm')



Outliers encontrados:
Empty DataFrame
Columns: [estacion, provincia, altura, numero, NroOACI, LAT_decimal, LON_decimal, mes_txt, prec_sup_1mm, humedad, nubosidad, prec_mm, temp, temp_max, temp_min, viento]
Index: []
Outliers encontrados:
Empty DataFrame
Columns: [estacion, provincia, altura, numero, NroOACI, LAT_decimal, LON_decimal, mes_txt, prec_sup_1mm, humedad, nubosidad, prec_mm, temp, temp_max, temp_min, viento]
Index: []


---
## EDA (Exploratory Data Analysis)
---

**18. Luego de calcular el coeficiente de correlación de Spearman entre la altura de las estaciones y la temperatura media para la región elegida, concluimos que:**


In [65]:
# Agrupamos por estación para obtener la altura (que es constante) y la temperatura media
df_estaciones = df.groupby('estacion')[['altura', 'temp']].mean().reset_index()

# Calculamos el coeficiente de correlación de Spearman
coef_spearman = df_estaciones['altura'].corr(df_estaciones['temp'], method='spearman')

print(f"Coeficiente de correlación de Spearman: {coef_spearman:.4f}")

Coeficiente de correlación de Spearman: -0.0818


**Conclusión:**
El coeficiente de correlación de Spearman nos da un valor de aproximadamente **-0.08**. Al ser un valor tan cercano a cero, **concluimos que no existe una relación monótona (ni lineal) significativa** entre la altura de las estaciones meteorológicas y la temperatura media en la región del NEA. Es decir, a diferencia de otras regiones donde a mayor altitud hace más frío, en esta región la altura no parece ser un factor determinante para las variaciones de temperatura media.

**Basándote en el valor de eta cuadrado para las variables provincia y precipitación de la región elegida, ¿qué podrías afirmar?**


In [66]:
# Eliminamos los valores nulos en las columnas que vamos a analizar
df_clean = df.dropna(subset=['prec_mm', 'provincia'])

# 1. Media global de precipitaciones
media_global = df_clean['prec_mm'].mean()

# 2. Suma de Cuadrados Total (SST)
sst = ((df_clean['prec_mm'] - media_global) ** 2).sum()

# 3. Suma de Cuadrados Entre grupos (SSB - Between groups)
agrupado = df_clean.groupby('provincia')['prec_mm']
ssb = ((agrupado.mean() - media_global) ** 2 * agrupado.count()).sum()

# 4. Eta Cuadrado
eta_cuadrado = ssb / sst

print(f"Valor de Eta Cuadrado: {eta_cuadrado:.4f}")

Valor de Eta Cuadrado: 0.2664


**Conclusión:**
El valor de **Eta Cuadrado ($\eta^2$) es de aproximadamente 0.266**.

Esta métrica nos indica qué porcentaje de la variación en una variable numérica (precipitación) es explicada por la variable categórica (provincia). Al obtener un **26.6%**, estamos ante un tamaño de efecto grande (usualmente valores mayores a 0.14 se consideran grandes).

Por lo tanto, **podemos afirmar que la provincia es un factor importante y explica una porción significativa de las diferencias de precipitación** en la región del NEA. Efectivamente, llueve distinto dependiendo de la provincia en la que estemos dentro de la región.

**Elegí una provincia de la región y describí brevemente su clima utilizando al menos tres variables del dataset (temperatura, precipitación, humedad, nubosidad o viento).**


In [ ]:
# Filtramos los datos únicamente para la provincia elegida (FORMOSA)
df_formosa = df[df['provincia'] == 'FORMOSA']

# Calculamos las métricas principales para poder describirlas
temp_media = df_formosa['temp'].mean()
temp_maxima = df_formosa['temp_max'].max()  # Obtenemos la temperatura máxima histórica
humedad_media = df_formosa['humedad'].mean()
prec_media = df_formosa['prec_mm'].mean()

print(f"--- Estadísticas de FORMOSA ---")
print(f"Temperatura media: {temp_media:.2f} °C")
print(f"Temperatura máxima registrada: {temp_maxima:.2f} °C")
print(f"Humedad media: {humedad_media:.2f} %")
print(f"Precipitación media: {prec_media:.2f} mm")

**Respuesta:**

Para este análisis he elegido la provincia de **Formosa**, la cual se caracteriza por un clima decididamente cálido y marcadamente húmedo, típico de su entorno subtropical dentro del NEA.

Analizando los registros del dataset, podemos observar que Formosa presenta una **temperatura media** elevada de aproximadamente 22.7 °C, reflejando un ambiente caluroso que se mantiene constante a lo largo del tiempo (con registros históricos de picos máximos que superan los 35 °C). Este calor está estrechamente acompañado por una **humedad relativa** media bastante alta, situándose cerca del 72.2%, lo cual contribuye a elevar significativamente la sensación térmica habitual en la provincia. Finalmente, en cuanto al régimen pluviométrico, las estaciones registran una **precipitación media** de aproximadamente 98.9 mm; estas lluvias regulares son fundamentales para el ecosistema de la zona y se combinan con una nubosidad parcial moderada.

**Análisis de meses con mayores temperaturas y precipitaciones en la región NEA**


In [70]:
# Agrupamos los datos por mes y calculamos la media de temperatura y precipitación
df_meses = df.groupby('mes_txt')[['temp', 'prec_mm']].mean().reset_index()

# Ordenamos para ver los meses con mayor temperatura
meses_mas_calurosos = df_meses.sort_values(by='temp', ascending=False).head(3)

# Ordenamos para ver los meses con mayor precipitación
meses_mas_lluviosos = df_meses.sort_values(by='prec_mm', ascending=False).head(3)

print("--- Top 3 Meses Más Calurosos ---")
print(meses_mas_calurosos[['mes_txt', 'temp']].to_string(index=False))
print("\n--- Top 3 Meses Más Lluviosos ---")
print(meses_mas_lluviosos[['mes_txt', 'prec_mm']].to_string(index=False))

--- Top 3 Meses Más Calurosos ---
mes_txt      temp
    Ene 26.500000
    Feb 25.690000
    Dic 25.581818

--- Top 3 Meses Más Lluviosos ---
mes_txt  prec_mm
    Dic   172.78
    Oct   171.98
    Ene   167.47


**Conclusión:**
Al observar los promedios mensuales en la región del NEA:
*   **Temperaturas:** Los meses más calurosos son los del verano (Enero, Febrero y Diciembre), alcanzando promedios por encima de los 25.5 °C.
*   **Precipitaciones:** Los meses más lluviosos se distribuyen entre fines de primavera y verano (Diciembre, Octubre y Enero), superando los 167 mm promedio mensuales. Esto es consistente con el clima subtropical, donde la temporada de lluvias suele coincidir con las épocas más cálidas del año.